In [0]:
# 1. Download, unzip, and list AIS data for June 01–07, 2023 (Python with retry & timeout)
import os
import time
import zipfile
import urllib.request
import urllib.error

DAYS = [1, 2, 3,]# 4, 5, 6, 7]
BASE_URL = "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_0{}.zip"
TMP_DIR = "/tmp"
MAX_RETRIES = 4
RETRY_DELAY = 10  # seconds between retries
TIMEOUT = 300      # seconds per download attempt


def download_with_retry(url: str, dest: str, retries: int = MAX_RETRIES,
                        delay: int = RETRY_DELAY, timeout: int = TIMEOUT) -> bool:
    """Download a URL to ``dest`` with retry, backoff, and timeout."""
    for attempt in range(1, retries + 1):
        try:
            print(f"  Attempt {attempt}/{retries} → {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as resp, open(dest, "wb") as f:
                f.write(resp.read())
            size_mb = os.path.getsize(dest) / (1024 * 1024)
            print(f"  ✓ Downloaded {dest} ({size_mb:.1f} MB)")
            return True
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, OSError) as exc:
            print(f"  ✗ Attempt {attempt} failed: {exc}")
            if attempt < retries:
                time.sleep(delay)
    print(f"  ✗ Giving up on {url} after {retries} attempts.")
    return False


def unzip_file(zip_path: str, extract_dir: str) -> None:
    """Extract ``zip_path`` into ``extract_dir`` (creates dir if needed)."""
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print(f"  ✓ Extracted to {extract_dir}")


# --- Download & unzip each day ---
for day in DAYS:
    url = BASE_URL.format(day)
    zip_path = os.path.join(TMP_DIR, f"AIS_2023_06_{day}.zip")
    extract_dir = os.path.join(TMP_DIR, f"AIS_2023_06_{day}")

    print(f"\n--- Day {day:02d} ---")
    if download_with_retry(url, zip_path):
        unzip_file(zip_path, extract_dir)
    # Clean up the zip to save space
    if os.path.exists(zip_path):
        os.remove(zip_path)

# --- List all extracted files ---
print("\n--- Extracted files ---")
for day in DAYS:
    extract_dir = os.path.join(TMP_DIR, f"AIS_2023_06_{day}")
    if os.path.isdir(extract_dir):
        for fname in sorted(os.listdir(extract_dir)):
            fpath = os.path.join(extract_dir, fname)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"{fpath}  ({size_mb:.1f} MB)")

In [0]:
!ls /tmp/AIS_2023_06_*

In [0]:
## Fisrt Check of expected size

def peso(ruta):
    total = 0
    for p in dbutils.fs.ls(ruta):
        total += peso(p.path) if p.isDir() else p.size
    return total

def humano(n):
    for unidad in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:,.1f} {unidad}"
        n /= 1024
    return f"{n:,.1f} TB"

import shutil

for DAY in DAYS:
    try :
        print(f"/tmp/AIS_2023_06_{DAY}/AIS_2023_06_0{DAY}.csv")
        shutil.copy(f"/tmp/AIS_2023_06_{DAY}/AIS_2023_06_0{DAY}.csv", f"/Volumes/ocean_watch/raw/ais_raw/ais_2023_06_0{DAY}")
        print(f"DAY {DAY} {humano(peso(f'/Volumes/ocean_watch/raw/ais_raw/ais_2023_06_{DAY}'))}")

    except Exception as e:
        print(e)
# DBTITLE 1,Read all files

Ahroa vamos a comprobar si los archivos tienen el esquema esperado. Los tipos de datos son los que se publican en el diccionario de datos.
Todas las columnas se han dejado como nullable True ya que son lecturas de sensores y contienen datos imperfectos, las posteriores limpiezas se encargaran de imputar valores validos, pero no podemos rechazar la lectura en esta etapa enforzando una tabla sin valores nulos

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

expected = [
    "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
    "VesselName", "IMO", "CallSign", "VesselType", "Status",
    "Length", "Width", "Draft", "Cargo", "TransceiverClass",
]

ais_schema = StructType([
    StructField("MMSI",            StringType(),   nullable=True),   # 9-digit vessel identifier — no math on IDs, string lets you check "not 9 digits"
    StructField("BaseDateTime",    TimestampType(),nullable=True),   # full UTC timestamp of the position report
    StructField("LAT",             DoubleType(),   nullable=True),   # decimal degrees, valid range [-90, 90]
    StructField("LON",             DoubleType(),   nullable=True),   # decimal degrees, valid range [-180, 180]
    StructField("SOG",             DoubleType(),   nullable=True),   # speed over ground, knots (has decimals)
    StructField("COG",             DoubleType(),   nullable=True),   # course over ground, degrees 0.0–359.9
    StructField("Heading",         IntegerType(),  nullable=True),   # degrees 0–359; 511 = "not available" per AIS standard
    StructField("VesselName",      StringType(),   nullable=True),   # free text, often null/blank
    StructField("IMO",             StringType(),   nullable=True),   # IMO number, appears as "IMO9074729" or blank — not numeric
    StructField("CallSign",        StringType(),   nullable=True),   # radio call sign, alphanumeric
    StructField("VesselType",      IntegerType(),  nullable=True),   # AIS ship-type code (30s=fishing, 70s=cargo, 80s=tanker...); some 4-digit codes exist
    StructField("Status",          IntegerType(),  nullable=True),   # navigational status 0–15, frequently blank
    StructField("Length",          DoubleType(),   nullable=True),   # meters
    StructField("Width",           DoubleType(),   nullable=True),   # meters
    StructField("Draft",           DoubleType(),   nullable=True),   # meters, one decimal
    StructField("Cargo",           IntegerType(),  nullable=True),   # cargo-type code, mostly empty
    StructField("TransceiverClass",StringType(),   nullable=True),   # "A" or "B"
])

Ahora creamos una tabla en el volumen para almacenar todos los datos y particionamos por dia.... 

In [0]:

spark.sql("CREATE OR REPLACE TABLE OCEAN_WATCH.RAW.AIS (MMSI STRING, BaseDateTime TIMESTAMP, LAT DOUBLE, LON DOUBLE, SOG DOUBLE, COG DOUBLE, Heading DOUBLE, NavStatus STRING, IMO STRING, CallSign STRING, VesselName STRING, Cargo STRING, VesselType STRING, Length DOUBLE, Width DOUBLE, Draft DOUBLE, TransceiverClass STRING)");

from pyspark.sql.functions import col

for DAY in DAYS:
    try :
        df = spark.read.csv(f"/Volumes/ocean_watch/raw/ais_raw/ais_2023_06_{DAY}", header=True, 
                       schema=ais_schema)
        df = df.withColumn("day", col("BaseDateTime").cast("date"))
        df.write.mode("append").saveAsTable("ais")
        print(f"DAY {DAY} {humano(peso(f'/Volumes/ocean_watch/raw/ais_raw/ais_2023_06_{DAY}.csv'))}")
        
    except Exception as e:
        print(e)
display(spark.sql("select count(*) from ais"))
display(spark.sql("select day, count(*) from ais group by day"))

#TODO fix errors and check why ais table is adding instead of being recreated??? 